In [2]:
import glob
import pandas as pd

MODEL = "ST12000NM0008"
SMART = [1, 3, 4, 5, 7, 9, 10, 12, 18, 187, 188, 190, 192, 193,
         194, 195, 197, 198, 199, 200, 240, 241, 242]
keep = ["date", "serial_number", "model", "capacity_bytes", "failure"] \
       + [f"smart_{n}_raw" for n in SMART]

files = sorted(glob.glob("../data/raw/**/*.csv", recursive=True))
print(len(files), "files")

181 files


In [3]:
parts = []
for f in files:
    d = pd.read_csv(f, usecols=keep)
    parts.append(d[d["model"] == MODEL])

data = pd.concat(parts, ignore_index=True)
data["date"] = pd.to_datetime(data["date"])
data = data.sort_values(["serial_number", "date"]).reset_index(drop=True)
print(data.shape)

(3335615, 28)


In [5]:
data.to_csv("../data/processed/st12000nm0008_q2_2026.csv.gz", index=False)

print("rows:", len(data))
print("drives:", data["serial_number"].nunique())
print("failures:", int(data["failure"].sum()))
print("dates:", data["date"].min().date(), "to", data["date"].max().date())

days_per_drive = data.groupby("serial_number").size()
print(days_per_drive.describe())
print("drives with a failure and their history length:")
failed = data.loc[data["failure"] == 1, "serial_number"]
print(days_per_drive[failed].describe())

rows: 3335615
drives: 18652
failures: 274
dates: 2026-01-01 to 2026-06-30
count    18652.000000
mean       178.834173
std         14.056070
min          1.000000
25%        181.000000
50%        181.000000
75%        181.000000
max        181.000000
dtype: float64
drives with a failure and their history length:
count    274.000000
mean      78.248175
std       52.210448
min        1.000000
25%       31.250000
50%       71.000000
75%      120.000000
max      181.000000
dtype: float64


In [6]:
hist = days_per_drive[failed]
for w in [7, 14, 21, 30]:
    print(f"history >= {w} days:", int((hist >= w).sum()), "failed drives")

print(data.loc[data["failure"] == 1, "date"].dt.to_period("M").value_counts().sort_index())

history >= 7 days: 252 failed drives
history >= 14 days: 239 failed drives
history >= 21 days: 223 failed drives
history >= 30 days: 207 failed drives
date
2026-01    49
2026-02    34
2026-03    46
2026-04    56
2026-05    54
2026-06    35
Freq: M, Name: count, dtype: int64


In [7]:
print(data["date"].min().date(), data["date"].max().date())
print(data.shape, data["serial_number"].nunique(), int(data["failure"].sum()))

2026-01-01 2026-06-30
(3335615, 28) 18652 274


In [9]:
data.to_csv("../data/processed/st12000nm0008_2026q1q2.csv.gz", index=False)